In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys
from tqdm import tqdm

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT)) 

from roi_classifier.prepare_data import prepare_roi_data
from roi_classifier.annotate_data import annotate_rois
from roi_classifier.train_classifier import train_roi_classifier




In [2]:
DATASET_ROOT = Path(r"C:\Users\mzinn1\Desktop\Morgan 1-20-26")  # TODO: set this to your data path
assert DATASET_ROOT.exists(), f"Dataset root {DATASET_ROOT} does not exist."

ROI_DIR = PROJECT_ROOT / "data"
ROI_DIR.mkdir(parents=True, exist_ok=True)

ROI_DATA_PATH = ROI_DIR / "all_roi_features.npy"

MODEL_OUT_DIR = PROJECT_ROOT / "models"
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = PROJECT_ROOT / "config/classifier_config.yaml"

print(f"Extracting fluorescence data from {DATASET_ROOT.__str__()}")
print(f"Saving engineered data to {ROI_DATA_PATH.__str__()}")
print(f"Saving models to {MODEL_OUT_DIR.__str__()}")
print(f"Configuring classifier according to {CONFIG_PATH.__str__()}")

Extracting fluorescence data from C:\Users\mzinn1\Desktop\Morgan 1-20-26
Saving engineered data to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy
Saving models to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models
Configuring classifier according to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\classifier_config.yaml


In [3]:
update = True # Change to false 
backup = False # Change as you wish; controls whether or not a backup of the original engineered data is saved


roi_data = prepare_roi_data(
    dataset_root=DATASET_ROOT,
    input_file=ROI_DATA_PATH,
    output_file=ROI_DATA_PATH,
    update=update,
    backup=backup
)



  ROI Summary
  Total rois: 18652
  Good: 573 | Bad: 280 | Unlabeled: 17799
  Manual: 853 | Auto: 0
  Total spikes stored: 10329


Updated 18652 ROIs
  - Preserved 853 manual labels
  - Preserved 10329 spikes

Saved 18652 ROIs to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy


In [4]:
unlabeled_only = True # Change as needed; if True, GUI only presents heretofore unlabeled ROIs
labeled_only = False # Change as needed; if True, GUI presents labeled ROIs, allowing you to review your work

assert not (unlabeled_only and labeled_only), "unlabeled_only and labeled_only cannot both be True — pick one or set both to False to show all ROIs."

n_annotations = 1000  # adjust as needed; you can always save your progress and quit mid-session
annotate_rois(data_path=ROI_DATA_PATH,
              n_annotations=n_annotations,
              unlabeled_only=True)


Loaded 18652 ROIs from C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy
Found 1000 unlabeled ROIs out of 17799 ROIs.
Session ended by user. Saving progress...
Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy
Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\all_roi_features.npy

  ROI Annotation Summary
  Total:     1000
  Labeled:   0
  Updated:   0
  Confirmed: 0
  Skipped:   0


  ROI Summary
  Total rois: 18652
  Good: 573 | Bad: 280 | Unlabeled: 17799
  Manual: 853 | Auto: 0
  Total spikes stored: 10329



In [5]:
name = "roi_classifier" # TODO Change this as needed for your own experimental/organizational needs 

results = train_roi_classifier(config_path=CONFIG_PATH, data_path=ROI_DATA_PATH, name="roi_classifier",
                     output_dir=MODEL_OUT_DIR, verbose=True, manual_only=True, overwrite=False)

Dataset Summary
--------------------------------------------------
Total labeled datapoints: 853
  Train: 682 | Test: 171

Label distribution:
              Bad (0)  Good (1)
  Train           224       458
  Test             56       115
  Total           280       573

Training on: Manual labels only


Exception ignored in: <function Image.__del__ at 0x000001F2C5662320>
Traceback (most recent call last):
  File "c:\Users\mzinn1\AppData\Local\anaconda3\envs\gcamp\lib\tkinter\__init__.py", line 4056, in __del__
    self.tk.call('image', 'delete', self.name)
RuntimeError: main thread is not in main loop



--------------------------------------------------
TUNED MODEL SUMMARY
--------------------------------------------------
Model:     RandomForestClassifier
Transform: raw
Features:  ['peak_density', 'derivative_skew', 'ac_decay', 'range_trace', 'var_of_var']

Hyperparameters:
  class_weight: None
  max_depth: None
  min_samples_leaf: 2
  min_samples_split: 2
  n_estimators: 100

Metrics:
  CV Accuracy:   0.9692
  Test Accuracy: 0.9766
  ROC AUC:       0.9972
  F1:            0.9767
  Precision:     0.9771
  Recall:        0.9766

Confusion Matrix:
              Pred 0  Pred 1
  Actual 0    55      1      
  Actual 1    3       112    
--------------------------------------------------
Saved model to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models\roi_classifier.joblib
Saved results to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models\roi_classifier_results.json
Saved results to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models
